# CRISE-ID: Contrastive RISE saliency maps
Runs `crise.run_crise` over all valid probes and caches saliency maps
to `results/crise_maps/`. Results are directly comparable to
`results/rise_alignedchip_baseline_multi/` from Rise_Baseline.ipynb.

In [ ]:
import os, json, random
import numpy as np

# ---- dataset/cache paths ----
SPLIT_PATH = "splits/lfw_1N_split.json"
G_IDS_PATH = "cache/gallery_ids.npy"
G_EMB_PATH = "cache/G.npy"

# ---- experiment control ----
MASTER_SEED       = 123
K_EXPERIMENTS     = 1680
MAX_PROBES_PER_ID = 1

# ---- RISE hyperparams (must match baseline for fair comparison) ----
N  = 1000
s  = 8
p1 = 0.5
BATCH_SAVE = 50

# ---- output ----
OUT_DIR = "results/crise_maps"
os.makedirs(OUT_DIR, exist_ok=True)

with open(SPLIT_PATH) as f:
    split = json.load(f)

gallery = split["gallery"]
probes  = split["probes"]

gallery_ids = np.load(G_IDS_PATH, allow_pickle=True).tolist()
G = np.load(G_EMB_PATH).astype(np.float32)
id_to_index = {pid: i for i, pid in enumerate(gallery_ids)}

print("Gallery embeddings:", G.shape)
print("Identities:", len(gallery))

In [ ]:
import onnxruntime as ort
from insightface.app import FaceAnalysis

app = FaceAnalysis(
    name="buffalo_l",
    providers=["CUDAExecutionProvider", "CPUExecutionProvider"]
)
app.prepare(ctx_id=0, det_size=(640, 640))
rec = app.models["recognition"]
print("ArcFace ready.")

In [ ]:
import importlib
import rise, crise
importlib.reload(rise)
importlib.reload(crise)

from crise import run_crise, TAU

print(f"TAU = {TAU}")

In [ ]:
import pandas as pd

valid_ids = [
    pid for pid in probes
    if pid in gallery and pid in id_to_index and len(probes[pid]) > 0
]
valid_ids = sorted(valid_ids)

sel_rng = random.Random(MASTER_SEED)
K = min(K_EXPERIMENTS, len(valid_ids))
picked_ids = sel_rng.sample(valid_ids, K)

results = []
for exp_i, pid in enumerate(picked_ids):
    probe_list = list(probes[pid])
    sel_rng.shuffle(probe_list)
    probe_list = probe_list[:max(1, MAX_PROBES_PER_ID)]

    for k, probe_path in enumerate(probe_list):
        # Same seed formula as baseline — different run_name_prefix keeps caches separate
        run_seed = MASTER_SEED * 10_000 + exp_i * 100 + k

        try:
            out = run_crise(
                true_id=pid,
                probe_path=probe_path,
                G=G,
                id_to_index=id_to_index,
                rec=rec,
                app=app,
                run_seed=run_seed,
                out_dir=OUT_DIR,
                tau=TAU,
                N=N, s=s, p1=p1,
                batch_save=BATCH_SAVE,
            )
            results.append({
                "true_id":       out["true_id"],
                "probe_file":    os.path.basename(out["probe_path"]),
                "probe_path":    out["probe_path"],
                "run_seed":      out["run_seed"],
                "failures":      out["failures"],
                "w_clean":       out["w_clean"],
                "w_black":       out["w_black"],
                "saliency_path": out["saliency_path"],
            })
        except Exception as e:
            results.append({
                "true_id":       pid,
                "probe_file":    os.path.basename(probe_path),
                "probe_path":    probe_path,
                "run_seed":      run_seed,
                "failures":      None,
                "w_clean":       None,
                "w_black":       None,
                "saliency_path": None,
                "error":         repr(e),
            })
            print("[error]", pid, probe_path, "->", repr(e))

df = pd.DataFrame(results)
df

In [ ]:
# Save summary CSV
summary_path = os.path.join(OUT_DIR, f"summary_crise_tau{TAU}_N{N}_s{s}_p{p1}_MASTERSEED{MASTER_SEED}.csv")
df.to_csv(summary_path, index=False)
print("Saved:", summary_path)

ok = df["saliency_path"].notna().sum()
total = len(df)
print(f"Completed: {ok} / {total}")
if "error" in df.columns:
    print(f"Errors: {df['error'].notna().sum()}")